In [2]:

import nltk
import spacy

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline



nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kartik/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kartik/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/kartik/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/kartik/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return " ".join(lemmatized)


tom hank star amazing film set new york


In [5]:
def get_entities(text):
     doc = nlp(text)
     return [(ent.text, ent.label_) for ent in doc.ents]


print(get_entities("Tom Hanks stars in this amazing film set in New York."))


[('Tom Hanks', 'PERSON'), ('New York', 'GPE')]


In [9]:
def get_sentiment(text):
    sia = SentimentIntensityAnalyzer()
    score = sia.polarity_scores(text)
    if score['compound'] >= 0.05:
        return "Positive"
    elif score['compound'] <= -0.05:
        return "Negative"
    else:
        return "Neutral"

# Test it
print(get_sentiment("Tom Hanks stars in this amazing film set in New York!"))

Positive


In [12]:
def spam_classifier():
        texts = [
            "Win free money now click here",
            "You have won a prize claim it now",
            "Free offer limited time click",
            "Hey are we meeting tomorrow?",
            "Can you send me the report please",
            "Lets catch up for lunch today"
        ]

        labels = ["spam", "spam", "spam", "ham", "ham", "ham"]

        
        model = Pipeline([
            ('tfidf', TfidfVectorizer()),
            ('classifier', MultinomialNB())
        ])

        model.fit(texts, labels)

        
        test = ["Free money click here now"]
        print(model.predict(test))
        return model


# Train the model
spam_model = spam_classifier()
print(spam_model.predict(["Free money click here now"]))

['spam']
['spam']


In [13]:
def analyze(text):
    preprocessed = preprocess(text)
    entities = get_entities(text)
    sentiment = get_sentiment(text)
    spam_prediction = spam_model.predict([text])[0]
    
    print("Preprocessed:", preprocessed)
    print("Entities:", entities)
    print("Sentiment:", sentiment)
    print("Spam Prediction:", spam_prediction)



analyze("Tom Hanks stars in this amazing film set in New York. Absolutely wonderful! Click here for free tickets.")


Preprocessed: tom hank star amazing film set new york absolutely wonderful click free ticket
Entities: [('Tom Hanks', 'PERSON'), ('New York', 'GPE')]
Sentiment: Positive
Spam Prediction: spam
